<a href="https://colab.research.google.com/github/buse-toklu/CENG467_Midterm/blob/main/q3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

# CELL 1: Install all required packages
!pip install datasets transformers torch sumy nltk rouge-score \
             sacrebleu bert-score nltk sentencepiece evaluate -q

import nltk
nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("omw-1.4")
nltk.download("averaged_perceptron_tagger")
nltk.download("averaged_perceptron_tagger_eng")

print(" All packages installed and NLTK data downloaded!")

In [ ]:

# CELL 2: All imports

import warnings, textwrap, re
warnings.filterwarnings("ignore")

import torch
import pandas as pd
import numpy as np

# Dataset
from datasets import load_dataset

# Summarization
from transformers import pipeline, AutoTokenizer, AutoModelForSeq2SeqLM
from sumy.parsers.plaintext import PlaintextParser
from sumy.nlp.tokenizers import Tokenizer
from sumy.summarizers.text_rank import TextRankSummarizer
from sumy.nlp.stemmers import Stemmer
from sumy.utils import get_stop_words

# Metrics
from rouge_score import rouge_scorer
import sacrebleu                        # BLEU
from nltk.translate.meteor_score import meteor_score
from nltk.tokenize import word_tokenize
from bert_score import score as bert_score_fn

print(" All imports successful!")
print(f" GPU available: {torch.cuda.is_available()}")

In [ ]:

# CELL 3: Load dataset — use 10 samples for robust evaluation

N_SAMPLES = 10

dataset = load_dataset("cnn_dailymail", "3.0.0", split="test")
subset  = dataset.select(range(N_SAMPLES))


lengths = [len(item["article"].split()) for item in subset]


print(f"  Full test split : {len(dataset):,} articles")
print(f" Evaluation subset size: {N_SAMPLES} samples")
print(f"  Avg article word count : {np.mean(lengths):.0f} words")
print(f"  Min / Max              : {min(lengths)} / {max(lengths)} words")

In [ ]:

# CELL 4: Extractive Summarization — TextRank


def textrank_summarize(text, num_sentences=3, language="english"):
    """
    Selects the most central sentences from the source text
    using a graph-based PageRank approach (no new words generated).
    """
    parser    = PlaintextParser.from_string(text, Tokenizer(language))
    stemmer   = Stemmer(language)
    summarizer = TextRankSummarizer(stemmer)
    summarizer.stop_words = get_stop_words(language)
    sentences = summarizer(parser.document, num_sentences)
    return " ".join(str(s) for s in sentences)


print("  Running TextRank on all samples...")
extractive_summaries = [textrank_summarize(item["article"]) for item in subset]
print(" Extractive summarization complete!\n")

for i, s in enumerate(extractive_summaries[:3]):
    print(f"── Sample {i+1} TextRank ──")
    print(textwrap.fill(s, 80), "\n")

In [ ]:

N_SAMPLES = 3

MODEL_NAME = "sshleifer/distilbart-cnn-12-6"
tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME)
model      = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
device     = "cpu"
model      = model.to(device)

def bart_summarize(text, max_length=130, min_length=30):
    inputs = tokenizer(
        text, return_tensors="pt",
        max_length=512,
        truncation=True
    ).to(device)
    summary_ids = model.generate(
        inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_length=max_length,
        min_length=min_length,
        num_beams=2,
        length_penalty=1.0,
        early_stopping=True,
        no_repeat_ngram_size=3,
    )
    return tokenizer.decode(summary_ids[0], skip_special_tokens=True)

print("Running BART on 3 samples...")
abstractive_summaries = [bart_summarize(subset[i]["article"]) for i in range(3)]
print("Done!")

In [ ]:
# CELL 6: Comprehensive metric evaluation

# Define references from the subset first.
# 'subset' contains 10 samples from CELL 3, but N_SAMPLES was set to 3 in CELL 5.
# This ensures 'references' is populated correctly from the dataset.
references = [item["highlights"] for item in subset]

# Truncate references, extractive_summaries, and abstractive_summaries to N_SAMPLES (which is 3 from CELL 5).
# This aligns the lists for metric computation.
references = references[:N_SAMPLES]
extractive_summaries = extractive_summaries[:N_SAMPLES]
abstractive_summaries = abstractive_summaries[:N_SAMPLES]

print(f"Ext: {len(extractive_summaries)}, Abs: {len(abstractive_summaries)}, Ref: {len(references)}")
# → Ext: 3, Abs: 3, Ref: 3 çıkmalı

rouge  = rouge_scorer.RougeScorer(["rouge1","rouge2","rougeL"], use_stemmer=True)

# ── Per-sample metric computation ─────────────────────────
def compute_all_metrics(generated_list, reference_list):
    """
    Returns a DataFrame with ROUGE-1/2/L, BLEU, METEOR, BERTScore
    for each sample, plus a final average row.
    """
    rows = []
    for i, (gen, ref) in enumerate(zip(generated_list, reference_list)):

        # ROUGE
        r = rouge.score(ref, gen)

        # BLEU  (sacrebleu expects list-of-references)
        bleu = sacrebleu.sentence_bleu(gen, [ref]).score / 100  # normalise 0–1

        # METEOR
        met = meteor_score([word_tokenize(ref)], word_tokenize(gen))

        rows.append({
            "Sample"  : i + 1,
            "ROUGE-1" : round(r["rouge1"].fmeasure,  4),
            "ROUGE-2" : round(r["rouge2"].fmeasure,  4),
            "ROUGE-L" : round(r["rougeL"].fmeasure,  4),
            "BLEU"    : round(bleu,                  4),
            "METEOR"  : round(met,                   4),
        })

    df = pd.DataFrame(rows)

    # BERTScore — run once in batch (faster)
    P, R, F = bert_score_fn(
        generated_list, reference_list,
        lang="en", verbose=False,
        device=device
    )
    df["BERTScore-F1"] = [round(f.item(), 4) for f in F]

    # Average row
    avg = df.drop(columns="Sample").mean().round(4).to_dict()
    avg["Sample"] = "AVG"
    df = pd.concat([df, pd.DataFrame([avg])], ignore_index=True)

    return df

# Compute metrics only once
print(" Computing metrics for TextRank (Extractive)...")
df_ext = compute_all_metrics(extractive_summaries, references)

print(" Computing metrics for BART (Abstractive)...")
df_abs = compute_all_metrics(abstractive_summaries, references)

print("\n All metrics computed!")

In [ ]:

# CELL 7: Display results tables
pd.set_option("display.float_format", "{:.4f}".format)
pd.set_option("display.max_columns", 10)
pd.set_option("display.width", 100)

print("=" * 70)
print("TextRank (Extractive) — Per-Sample Metrics")
print("=" * 70)
print(df_ext.to_string(index=False))

print("\n")

print("=" * 70)
print(" BART (Abstractive) — Per-Sample Metrics")
print("=" * 70)
print(df_abs.to_string(index=False))

# ── Head-to-head average comparison ───────────────────────
avg_ext = df_ext[df_ext["Sample"] == "AVG"].drop(columns="Sample").iloc[0]
avg_abs = df_abs[df_abs["Sample"] == "AVG"].drop(columns="Sample").iloc[0]

comparison = pd.DataFrame({
    "Metric"              : avg_ext.index,
    "TextRank (Extract.)" : avg_ext.values,
    "BART (Abstract.)"    : avg_abs.values,
    "Winner"              : [
        "TextRank" if e > a else "BART"
        for e, a in zip(avg_ext.values, avg_abs.values)
    ]
})

print("\n")
print("=" * 70)
print("Head-to-Head Average Comparison")
print("=" * 70)
print(comparison.to_string(index=False))

In [ ]:
# ============================================================
# CELL 8: Qualitative analysis — 3 annotated examples
# ============================================================

QUAL_INDICES = [0, 1, 2]   # pick any 3 samples from 0–9

def word_count(text):
    return len(text.split())

def score_sample(gen, ref):
    r  = rouge.score(ref, gen)
    b  = sacrebleu.sentence_bleu(gen, [ref]).score / 100
    m  = meteor_score([word_tokenize(ref)], word_tokenize(gen))
    return {
        "ROUGE-1": round(r["rouge1"].fmeasure, 3),
        "ROUGE-2": round(r["rouge2"].fmeasure, 3),
        "ROUGE-L": round(r["rougeL"].fmeasure, 3),
        "BLEU"   : round(b, 3),
        "METEOR" : round(m, 3),
    }

# BERTScore for the 3 qualitative samples (both methods)
_, _, F_ext = bert_score_fn(
    [extractive_summaries[i] for i in QUAL_INDICES],
    [references[i]           for i in QUAL_INDICES],
    lang="en", verbose=False, device=device
)
_, _, F_abs = bert_score_fn(
    [abstractive_summaries[i] for i in QUAL_INDICES],
    [references[i]            for i in QUAL_INDICES],
    lang="en", verbose=False, device=device
)

for rank, idx in enumerate(QUAL_INDICES):
    article   = subset[idx]["article"]
    reference = references[idx]
    ext_sum   = extractive_summaries[idx]
    abs_sum   = abstractive_summaries[idx]

    se = score_sample(ext_sum, reference)
    sa = score_sample(abs_sum, reference)
    se["BERTScore-F1"] = round(F_ext[rank].item(), 3)
    sa["BERTScore-F1"] = round(F_abs[rank].item(), 3)

    div = "═" * 72
    print(div)
    print(f"  QUALITATIVE EXAMPLE {rank+1}  (Dataset index {idx})")
    print(div)

    print("\n ARTICLE SNIPPET (first 400 chars):")
    print(textwrap.fill(article[:400], 72) + " …\n")

    print(" REFERENCE SUMMARY:")
    print(textwrap.fill(reference, 72))
    print(f"   [{word_count(reference)} words]\n")

    print("─" * 72)
    print(" TEXTRANK — Extractive Summary:")
    print(textwrap.fill(ext_sum, 72))
    print(f"   [{word_count(ext_sum)} words]")
    print("   Scores:", "  ".join(f"{k}={v}" for k,v in se.items()))

    print()
    print(" BART — Abstractive Summary:")
    print(textwrap.fill(abs_sum, 72))
    print(f"   [{word_count(abs_sum)} words]")
    print("   Scores:", "  ".join(f"{k}={v}" for k,v in sa.items()))

    print()
